In [ ]:
# import libraries
import pandas as pd, numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# import dataset
file_path = "../data/raw/Guns incident Data.csv"
df = pd.read_csv(file_path)

# basic cleaning (can remove and import cleaned dataset later)
df = df.replace({"NA": np.nan}) # replace NA
df["Age"] = pd.to_numeric(df["Age"], errors = "coerce")
df["Police involvement"] = pd.to_numeric(df["Police involvement"], errors="coerce")
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

# columns to keep
useful_cols = ["Date", "Reason", "Education", "Sex", "Age", "Race", "Place of incident", "Police involvement"]
df_copy = df[useful_cols].copy()

# --- Feature Engineering ---
# age profile
bins = [-np.inf, 17, 25, 35, 45, 55, 65, np.inf]
labels = ["<=17","18-25","26-35","36-45","46-55","56-65","66+"]
df_copy["AgeRange"] = pd.cut(df_copy["Age"], bins = bins, labels = labels)

# date features
df_copy["Month"]   = df_copy["Date"].dt.month
df_copy["Weekday"] = df_copy["Date"].dt.weekday
df_copy["IsWeekend"] = df_copy["Weekday"].isin([5,6]).astype("float")

# helper function to combine categories with small counts
# this is so that we avoid one-hot explosion, which adds noise and to avoid micro clusters
def combine_small_categories(s: pd.Series, min_count=700, other_label="Other"):
    counts = s.value_counts(dropna=True)
    keep = counts[counts >= min_count].index
    return s.where(s.isin(keep), other_label)

for col in ["Race","Education","Place of incident","Reason"]:
    df_copy[col] = combine_small_categories(df_copy[col], min_count=700)

# final feature set
cat_features = ["Race","Education","Place of incident","Reason","AgeBucket","Sex"] 
num_features = ["Age","Month","Weekday","IsWeekend","Police involvement"]

df_cluster = df_copy[cat_features + num_features].copy()

# --- Preprocess ---
"""
- cat_pipeline:
    * SimpleImputer(strategy="most_frequent"): fills missing categories with the mode so that the OneHotEncoder doesnt see the NaNs
    * OneHotEncoder(handle_unknown="ignore", sparse_output=False) expands categories into dummy columns
- num_pipeline:
    * SimpleImputer(median): fills missing numeric values robustly against outliers.
    * StandardScaler(): standardizes numerics (mean = 0, variance = 1)
- ColumnTransformer:
    * Applies cat_pipeline to cat_features and num_pipeline to num_features in a step
    * remainder = "drop": keeps only transformed columns (and drops anything that is not listed)
"""
cat_pipeline = Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")),
    ("oh" , OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

num_pipeline = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc" , StandardScaler())
])

pre = ColumnTransformer([
    ("cat", cat_pipe, feat_cat),
    ("num", num_pipe, feat_num)
], remainder="drop")

# --- Pick K (silhouette + elbow method) and fit K-Means---
df_pre = pre.fit_transform(df_cluster)

# SILHOUETTE SEARCH
best_k, best_score, scores = None, -1, {}
for k in range(2, 11):
    km = KMeans(n_clusters=k, n_init="auto", random_state=3244)
    labels = km.fit_predict(df_pre)
    s = silhouette_score(df_pre, labels)
    scores[k] = s
    if s > best_score:
        best_k, best_score = k, s
print("Best K:", best_k, "silhouette:", round(best_score,3))

# ELBOW (optional)
inertias = []
for k in range(2, 11):
    inertias.append(KMeans(n_clusters=k, n_init="auto", random_state=3244).fit(df_pre).inertia_)
plt.plot(range(2,11), inertias, marker="o"); plt.title("Elbow"); plt.xlabel("K"); plt.ylabel("Inertia"); plt.show()

# FINAL FIT
kmeans = KMeans(n_clusters=best_k, n_init="auto", random_state=3244)
labels = kmeans.fit_predict(df_pre)

# 2D PCA plot
pca = PCA(n_components=2, random_state=3244)
X_2d = pca.fit_transform(df_pre)
plot_df = pd.DataFrame({"PC1":X_2d[:,0], "PC2":X_2d[:,1], "cluster":labels})

for c in sorted(plot_df.cluster.unique()):
    sub = plot_df[plot_df.cluster==c]
    plt.scatter(sub.PC1, sub.PC2, s=8, alpha=0.7, label=f"C{c}")
plt.legend(); plt.title(f"PCA by K-Means clusters (K={best_k})")
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.tight_layout(); plt.show()

# --- Cluster Profile Table ---
prof = df_copy.copy()
prof["cluster"] = labels

# Numeric summary
num_summary = prof.groupby("cluster")[["Age","Police involvement"]].agg(["count","mean","median"]).round(2)

# Proportions per category
def prop_table(col):
    tab = (prof.pivot_table(index="cluster", columns=col, values="Age", aggfunc="size", fill_value=0)
              .apply(lambda r: r/r.sum(), axis=1))
    tab.columns = [f"{col}={c}" for c in tab.columns]
    return tab.round(3)

cat_summary = pd.concat([prop_table(c) for c in ["Race","Education","Place of incident","Reason","AgeBucket","Sex"]], axis=1)

cluster_profile = pd.concat([num_summary, cat_summary], axis=1).sort_index()
cluster_profile.head()
